# Kaggriculture — isolated frozen benchmark environment

This notebook uses separate, immutable checkouts for benchmark infrastructure, the A-v2 candidate, and Champion. It does not run a benchmark until all pre-flight and smoke-test checks pass.


In [ ]:
# 1. Create three independent, detached, immutable checkouts
from pathlib import Path
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/Pr1meGG/fieldops-kaggriculture.git'
INFRASTRUCTURE_COMMIT = '40032f1b4e5321306e273c21d4dec1f4eba1aea0'
CANDIDATE_COMMIT = '499d043e81041152b1422e95cb1515a6d0476b8a'
CHAMPION_COMMIT = '1b05b9b6e4932e6bdf8a01497cf94cfbbd9aa61f'
RUNNER_PATH = 'research/benchmark_runner.py'
WORKING = Path('/kaggle/working')
INFRASTRUCTURE = WORKING / 'benchmarkinfrastructure'
CANDIDATE = WORKING / 'candidate'
CHAMPION = WORKING / 'champion'

def run(*args):
    return subprocess.run(args, check=True, text=True, capture_output=True).stdout.strip()

def checkout(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    # Fetch all advertised refs only to obtain the immutable commit below.
    run('git', 'clone', '--no-single-branch', REPOSITORY_URL, str(destination))
    run('git', '-C', str(destination), 'checkout', '--detach', commit)
    actual = run('git', '-C', str(destination), 'rev-parse', 'HEAD')
    assert actual == commit, f'{destination.name} HEAD {actual} != {commit}'

checkout(INFRASTRUCTURE, INFRASTRUCTURE_COMMIT)
checkout(CANDIDATE, CANDIDATE_COMMIT)
checkout(CHAMPION, CHAMPION_COMMIT)

# The harness is sourced by Git from the frozen candidate tree, where its exact blob lives.
# This adds no candidate agent code to the infrastructure checkout and leaves its HEAD pinned.
run('git', '-C', str(INFRASTRUCTURE), 'checkout', CANDIDATE_COMMIT, '--', RUNNER_PATH)
expected_runner_blob = run('git', '-C', str(INFRASTRUCTURE), 'rev-parse',
                          f'{CANDIDATE_COMMIT}:{RUNNER_PATH}')
actual_runner_blob = run('git', '-C', str(INFRASTRUCTURE), 'hash-object',
                         str(INFRASTRUCTURE / RUNNER_PATH))
assert actual_runner_blob == expected_runner_blob, 'benchmark runner blob mismatch'
print('Created isolated benchmarkinfrastructure/, candidate/, and champion/ checkouts.')


In [ ]:
# 2. Immutable checkout pre-flight: fail before any seed can execute
expected_heads = {
    'infrastructure': (INFRASTRUCTURE, INFRASTRUCTURE_COMMIT),
    'candidate': (CANDIDATE, CANDIDATE_COMMIT),
    'champion': (CHAMPION, CHAMPION_COMMIT),
}
for label, (directory, expected) in expected_heads.items():
    actual = run('git', '-C', str(directory), 'rev-parse', 'HEAD')
    print(f'{label} HEAD = {actual}')
    assert actual == expected, f'{label} HEAD {actual} != {expected}'

assert (INFRASTRUCTURE / RUNNER_PATH).is_file(), 'benchmark_runner.py is missing'
assert (CANDIDATE / 'src/fieldops/agent.py').is_file(), 'candidate agent is missing'
assert (CHAMPION / 'src/fieldops/agent.py').is_file(), 'champion agent is missing'
print('PRE-FLIGHT PASS — no benchmark seed has executed.')


In [ ]:
# 3. Kaggle-authoritative engine pre-flight: use the preinstalled runtime only
import kaggle_environments
from kaggle_environments import make

EXPECTED_ENGINE_MODULE_VERSION = '1.32.6'
EXPECTED_ENGINE_SPEC_VERSION = '0.1.0'
CONFIGURATION_FINGERPRINT = {
    'episodeSteps': 720,
    'townCenterSellInterval': 24,
    'farmHandCostMult': 1,
    'townShopSellInterval': 4,
    'townShopUnlockInterval': 3,
    'turnsPerDay': 24,
    'marketParams': {},
}

# `seed` is explicit for deterministic initialization; it is intentionally not part
# of the public configuration fingerprint because the engine records it in env.info.
engine_config = {**CONFIGURATION_FINGERPRINT, 'seed': 1}
print(f'kaggle_environments.__version__ = {kaggle_environments.__version__}')
environment = make('kaggriculture', debug=False, configuration=engine_config)
assert environment is not None, 'Kaggriculture environment did not initialize'
environment_json = environment.toJSON()

assert kaggle_environments.__version__ == EXPECTED_ENGINE_MODULE_VERSION, (
    f'kaggle_environments.__version__={kaggle_environments.__version__}, '
    f'expected {EXPECTED_ENGINE_MODULE_VERSION}'
)
assert environment_json['module_version'] == EXPECTED_ENGINE_MODULE_VERSION, (
    f'env module_version={environment_json["module_version"]}, '
    f'expected {EXPECTED_ENGINE_MODULE_VERSION}'
)
assert environment.version == EXPECTED_ENGINE_SPEC_VERSION, (
    f'env spec version={environment.version}, expected {EXPECTED_ENGINE_SPEC_VERSION}'
)
actual_fingerprint = {
    key: getattr(environment.configuration, key)
    for key in CONFIGURATION_FINGERPRINT
}
assert actual_fingerprint == CONFIGURATION_FINGERPRINT, (
    f'engine configuration {actual_fingerprint} != {CONFIGURATION_FINGERPRINT}'
)
assert environment.info['seed'] == 1, 'engine did not retain the explicit seed'
print('ENGINE PRE-FLIGHT PASS — Kaggle runtime matches the authoritative baseline.')


In [ ]:
# 4. Mandatory zero-seed smoke test and immutable manifest
import importlib
import importlib.util
import json
import sys

runner_spec = importlib.util.spec_from_file_location(
    'benchmark_runner_smoke', INFRASTRUCTURE / RUNNER_PATH
)
runner_module = importlib.util.module_from_spec(runner_spec)
runner_spec.loader.exec_module(runner_module)
assert callable(runner_module.run_experiment), 'benchmark runner did not import'

def import_isolated_agent(label, source_root):
    for name in list(sys.modules):
        if name == 'fieldops' or name.startswith('fieldops.'):
            del sys.modules[name]
    sys.path.insert(0, str(source_root))
    try:
        module = importlib.import_module('fieldops.agent')
        assert Path(module.__file__).resolve().is_relative_to(source_root.resolve())
        assert callable(module.agent), f'{label} agent is not callable'
        print(f'{label} agent import: {module.__file__}')
    finally:
        sys.path.remove(str(source_root))

import_isolated_agent('candidate', CANDIDATE / 'src')
import_isolated_agent('champion', CHAMPION / 'src')

BENCHMARK_MANIFEST = {
    'candidate_sha': CANDIDATE_COMMIT,
    'champion_sha': CHAMPION_COMMIT,
    'infrastructure_sha': INFRASTRUCTURE_COMMIT,
    'benchmark_runner_blob': actual_runner_blob,
    'kaggle_environments_version': kaggle_environments.__version__,
    'env_module_version': environment_json['module_version'],
    'env_spec_version': environment.version,
    'configuration_fingerprint': actual_fingerprint,
}
print(json.dumps(BENCHMARK_MANIFEST, indent=2, sort_keys=True))
PREFLIGHT_COMPLETE = True
print('DRY-RUN PASS — runner and both agents imported; 0 seeds run.')


In [ ]:
# 5. Benchmark A-v2 candidate against frozen Champion (50 seeds)
# HARD STOP: this cell fails unless every preceding pre-flight completed successfully.
assert PREFLIGHT_COMPLETE is True, 'FATAL: complete all pre-flight cells before running seeds'
!python /kaggle/working/benchmarkinfrastructure/research/benchmark_runner.py \
    --agent-root /kaggle/working/candidate/src \
    --agent fieldops.agent:agent \
    --champion-commit 1b05b9b6e4932e6bdf8a01497cf94cfbbd9aa61f \
    --opponent /kaggle/working/champion/src/fieldops/agent.py \
    --seeds 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 \
    --experiment-name a_v2_vs_champion_50seed \
    --results-dir /kaggle/working/results


In [ ]:
# 6. Package results (only after the benchmark cell has been intentionally run)
!cd /kaggle/working && tar -czvf results.tar.gz results/
print('Done! Download results.tar.gz from the output panel.')
